# 第8章 规划与学习

本章学习规划与学习的核心概念，包括Dyna-Q框架、优先级规划和MPC算法。

## 1. 环境配置

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("环境配置完成!")

## 2. 网格世界环境实现

In [ ]:
class GridWorldEnv:
    """网格世界环境"""
    def __init__(self, n=5):
        self.n = n
        self.goal = (n-1, n-1)
        self.obstacles = [(1, 1), (2, 2), (3, 1)]
        self.state = (0, 0)
        self.actions = [(0, -1), (0, 1), (-1, 0), (1, 0)]  # 左、右、上、下
    
    def reset(self):
        self.state = (0, 0)
        return self.state
    
    def step(self, action):
        dr, dc = self.actions[action]
        nr = max(0, min(self.n-1, self.state[0] + dr))
        nc = max(0, min(self.n-1, self.state[1] + dc))
        new_state = (nr, nc)
        
        if new_state in self.obstacles:
            new_state = (0, 0)  # 撞到障碍物回到起点
            reward = -1
        elif new_state == self.goal:
            reward = 10
        else:
            reward = -0.1
        
        done = new_state == self.goal
        self.state = new_state
        return new_state, reward, done
    
    def get_state_idx(self, state):
        return state[0] * self.n + state[1]

env = GridWorldEnv(5)
print(f"网格大小: {env.n}x{env.n}")
print(f"目标位置: {env.goal}")
print(f"障碍物位置: {env.obstacles}")
state = env.reset()
print(f"初始状态: {state}")

## 3. Dyna-Q算法实现

In [ ]:
class DynaQAgent:
    """Dyna-Q算法智能体"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}  # 环境模型: (state, action) -> (reward, next_state)
    
    def choose_action(self, state):
        """ε-greedy动作选择"""
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        else:
            return np.argmax(self.q_table[state])
    
    def learn(self, state, action, reward, next_state):
        """从真实经验学习 - Q学习更新"""
        target = reward + self.gamma * np.max(self.q_table[next_state])
        td_error = target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error
        
        # 更新环境模型
        self.model[(state, action)] = (reward, next_state)
        return td_error
    
    def plan(self, k=5):
        """从模拟经验规划"""
        for _ in range(k):
            if not self.model:
                break
            # 随机选择之前经历过的状态-动作对
            state, action = random.choice(list(self.model.keys()))
            reward, next_state = self.model[(state, action)]
            
            # Q学习更新
            target = reward + self.gamma * np.max(self.q_table[next_state])
            td_error = target - self.q_table[state, action]
            self.q_table[state, action] += self.alpha * td_error
    
    def reset(self):
        """重置智能体"""
        self.q_table = np.zeros((self.n_states, self.n_actions))
        self.model = {}

def train_dynaq(env, k=5, n_episodes=100, max_steps=50):
    """训练Dyna-Q智能体"""
    n_states = env.n * env.n
    n_actions = 4
    agent = DynaQAgent(n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=0.1)
    
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.choose_action(state_idx)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            # 从真实经验学习
            agent.learn(state_idx, action, reward, next_state_idx)
            
            # 从模拟经验规划
            agent.plan(k)
            
            total_reward += reward
            state_idx = next_state_idx
            
            if done:
                break
        
        rewards_history.append(total_reward)
    
    return rewards_history, agent

print("Dyna-Q算法实现完成")

## 4. 编程题1：验证规划次数k对样本效率的影响

### 题目描述
实现 Dyna-Q 算法，在网格世界任务中验证规划次数 k 对样本效率的影响，分析 k=0、k=5、k=10 时的收敛差异。

In [ ]:
def programming_exercise_1():
    """编程题1：验证规划次数k对样本效率的影响"""
    n_episodes = 100
    max_steps = 50
    k_values = [0, 5, 10]
    
    results = {}
    plt.figure(figsize=(12, 5))
    
    for k in k_values:
        print(f"训练 k={k}...")
        rewards_history, agent = train_dynaq(GridWorldEnv(5), k=k, n_episodes=n_episodes, max_steps=max_steps)
        results[k] = rewards_history
        
        # 原始曲线
        plt.subplot(1, 2, 1)
        plt.plot(rewards_history, alpha=0.3, label=f'k={k} 原始')
        # 平滑曲线
        window = 10
        smoothed = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
        plt.plot(smoothed, linewidth=2, label=f'k={k} 平滑')
    
    plt.xlabel('回合数')
    plt.ylabel('累积奖励')
    plt.title('编程题1: 规划次数k对收敛速度的影响')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 收敛后平均奖励对比
    plt.subplot(1, 2, 2)
    final_rewards = [np.mean(results[k][-20:]) for k in k_values]
    plt.bar([str(k) for k in k_values], final_rewards, color=['blue', 'orange', 'green'])
    plt.xlabel('规划次数 k')
    plt.ylabel('最后20回合平均奖励')
    plt.title('收敛性能对比')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 分析结论 ===")
    print("k=0: 无规划，纯真实经验学习，收敛最慢")
    print("k=5: 适度规划，收敛速度明显提升")
    print("k=10: 规划次数越多，收敛越快，但计算开销增大")
    
    return results

pe1_result = programming_exercise_1()

## 5. 优先级规划Dyna-Q算法

In [ ]:
class DynaQPriorityAgent:
    """带优先级规划的Dyna-Q算法"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=0.1, theta=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.theta = theta  # 优先级阈值
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}  # 环境模型
        self.priority_queue = {}  # 优先级队列: (state, action) -> td_error
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        else:
            return np.argmax(self.q_table[state])
    
    def learn(self, state, action, reward, next_state):
        """从真实经验学习"""
        target = reward + self.gamma * np.max(self.q_table[next_state])
        td_error = abs(target - self.q_table[state, action])
        self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])
        
        self.model[(state, action)] = (reward, next_state)
        
        # 加入优先级队列
        if td_error > self.theta:
            self.priority_queue[(state, action)] = td_error
    
    def plan(self, k=5):
        """优先级规划"""
        for _ in range(k):
            if not self.priority_queue:
                break
            # 选择优先级最高的状态-动作对
            state_action = max(self.priority_queue.keys(), key=lambda x: self.priority_queue[x])
            del self.priority_queue[state_action]
            state, action = state_action
            reward, next_state = self.model[state_action]
            
            target = reward + self.gamma * np.max(self.q_table[next_state])
            new_td_error = abs(target - self.q_table[state, action])
            self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])
            
            if new_td_error > self.theta:
                self.priority_queue[state_action] = new_td_error
    
    def reset(self):
        self.q_table = np.zeros((self.n_states, self.n_actions))
        self.model = {}
        self.priority_queue = {}

def train_dynaq_priority(env, n_episodes=100, max_steps=50, k=5):
    """训练优先级规划Dyna-Q"""
    n_states = env.n * env.n
    n_actions = 4
    agent = DynaQPriorityAgent(n_states, n_actions)
    
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.choose_action(state_idx)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            agent.learn(state_idx, action, reward, next_state_idx)
            agent.plan(k)
            
            total_reward += reward
            state_idx = next_state_idx
            
            if done:
                break
        
        rewards_history.append(total_reward)
    
    return rewards_history, agent

print("优先级规划Dyna-Q算法实现完成")

## 6. 编程题2：优先级规划与随机规划对比

### 题目描述
实现带优先级规划的 Dyna-Q 算法，比较优先级规划与随机规划在相同任务中的性能差异。

In [ ]:
def train_random_planning(env, n_episodes=100, max_steps=50, k=5):
    """随机规划Dyna-Q (对比基准)"""
    n_states = env.n * env.n
    n_actions = 4
    agent = DynaQAgent(n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=0.1)
    
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.choose_action(state_idx)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            agent.learn(state_idx, action, reward, next_state_idx)
            # 随机规划（与优先级规划对比）
            agent.plan(k)
            
            total_reward += reward
            state_idx = next_state_idx
            
            if done:
                break
        
        rewards_history.append(total_reward)
    
    return rewards_history

def programming_exercise_2():
    """编程题2：优先级规划与随机规划对比"""
    n_episodes = 100
    max_steps = 50
    k = 5
    
    print("训练随机规划...")
    random_rewards = train_random_planning(GridWorldEnv(5), n_episodes=n_episodes, max_steps=max_steps, k=k)
    
    print("训练优先级规划...")
    priority_rewards, _ = train_dynaq_priority(GridWorldEnv(5), n_episodes=n_episodes, max_steps=max_steps, k=k)
    
    window = 10
    random_smooth = np.convolve(random_rewards, np.ones(window)/window, mode='valid')
    priority_smooth = np.convolve(priority_rewards, np.ones(window)/window, mode='valid')
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(random_smooth, color='blue', linewidth=2, label='随机规划')
    plt.plot(priority_smooth, color='red', linewidth=2, label='优先级规划')
    plt.xlabel('回合数')
    plt.ylabel('平滑奖励')
    plt.title('编程题2: 规划方式对比')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    final_random = np.mean(random_rewards[-20:])
    final_priority = np.mean(priority_rewards[-20:])
    plt.bar(['随机规划', '优先级规划'], [final_random, final_priority], color=['blue', 'red'])
    plt.ylabel('最后20回合平均奖励')
    plt.title('收敛性能对比')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 分析结论 ===")
    print("优先级规划通过优先更新高TD误差的状态-动作对，更高效利用规划资源")
    print(f"随机规划: {final_random:.2f}, 优先级规划: {final_priority:.2f}")
    print("优先级规划通常比随机规划收敛更快")
    
    return random_rewards, priority_rewards

pe2_result = programming_exercise_2()

## 7. MPC算法实现

In [ ]:
class InvertedPendulumEnv:
    """倒立摆环境"""
    def __init__(self):
        self.state = np.array([0.0, 0.0])  # [角度, 角速度]
        self.dt = 0.05
        self.goal = 0.0  # 目标角度
    
    def reset(self):
        self.state = np.array([np.pi / 4, 0.0])  # 初始偏离
        return self.state.copy()
    
    def step(self, action):
        """倒立摆动力学"""
        theta, theta_dot = self.state
        u = action[0]  # 控制力
        
        # 简化动力学
        theta_dot += self.dt * (u - theta)  # 简化模型
        theta += self.dt * theta_dot
        
        self.state = np.array([theta, theta_dot])
        
        # 奖励: 靠近目标且消耗能量少
        reward = -abs(theta - self.goal) - 0.01 * u**2
        
        done = abs(theta) < 0.05
        return self.state.copy(), reward, done

class MPCAgent:
    """MPC控制器"""
    def __init__(self, env, H=5):
        self.env = env
        self.H = H  # 规划深度
        self.actions_candidates = np.linspace(-2, 2, 5)  # 动作候选
    
    def plan(self, state):
        """MPC规划: 滚动优化"""
        best_action = 0
        best_value = float('-inf')
        
        for u in self.actions_candidates:
            # 模拟H步
            sim_state = state.copy()
            value = 0
            for _ in range(self.H):
                theta, theta_dot = sim_state
                sim_state[1] += self.env.dt * (u - theta)
                sim_state[0] += self.env.dt * sim_state[1]
                value += -(abs(sim_state[0]) + 0.01 * u**2)
            
            if value > best_value:
                best_value = value
                best_action = u
        
        return np.array([best_action])

def train_mpc(H=5, n_steps=200):
    """训练MPC控制器"""
    env = InvertedPendulumEnv()
    mpc = MPCAgent(env, H=H)
    
    states = []
    rewards = []
    
    state = env.reset()
    for _ in range(n_steps):
        action = mpc.plan(state)
        next_state, reward, done = env.step(action)
        states.append(state[0])
        rewards.append(reward)
        state = next_state
        if done:
            break
    
    return states, rewards

print("MPC算法实现完成")

## 8. 编程题3：MPC规划深度H对控制精度的影响

### 题目描述
在连续控制任务（如倒立摆）中实现 MPC 算法，分析规划深度 H 对控制精度和实时性的影响。

In [ ]:
def programming_exercise_3():
    """编程题3：MPC规划深度H对控制精度的影响"""
    H_values = [1, 3, 5, 10]
    
    plt.figure(figsize=(14, 5))
    results = {}
    
    for idx, H in enumerate(H_values):
        print(f"训练 H={H}...")
        states, rewards = train_mpc(H=H, n_steps=200)
        results[H] = {'states': states, 'rewards': rewards}
        
        plt.subplot(1, 2, 1)
        plt.plot(states, label=f'H={H}', alpha=0.7)
        
        plt.subplot(1, 2, 2)
        plt.plot(rewards, label=f'H={H}', alpha=0.7)
    
    plt.subplot(1, 2, 1)
    plt.xlabel('时间步')
    plt.ylabel('角度')
    plt.title('编程题3: 不同H下的角度变化')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.xlabel('时间步')
    plt.ylabel('奖励')
    plt.title('不同H下的累积奖励')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 分析结论 ===")
    print("H=1: 规划深度最小，计算最快，但可能无法找到长期最优路径")
    print("H=3-5: 平衡了计算效率和控制精度")
    print("H=10: 规划深度最大，控制精度高，但计算开销大")
    print("实际应用中需根据实时性要求选择合适的H值")
    
    # 计算每个H的总奖励
    for H in H_values:
        total_reward = sum(results[H]['rewards'])
        print(f"H={H}: 总奖励={total_reward:.2f}")
    
    return results

pe3_result = programming_exercise_3()

## 9. 本章小结

本章介绍了规划与学习的核心概念:

1. **Dyna-Q框架**: 将真实学习与模拟规划结合，通过k次规划迭代提升样本效率
2. **优先级规划**: 优先更新高TD误差的状态-动作对，更高效利用规划资源
3. **MPC算法**: 滚动优化，在连续控制任务中通过模拟规划选择最优动作

### 编程题总结
- 编程题1: k值越大收敛越快，但计算开销增大
- 编程题2: 优先级规划优于随机规划
- 编程题3: H值选择需平衡精度和实时性